# Generate embeddings (Colab)

Runs `src/generate_embeddings.py`'s `run()` directly (no `!python ... --args` CLI) to compute image embeddings for a (model, dataset, split) combo and save them as `outputs/embeddings/<dataset>/<model>_<split>.npz` (`embeddings` float32, `labels` int64) — feeds `src/eval_embeddings.py` / `src/eval_online.py` / `src/eval_abtt.py`.

Registries (models: CLIP/DINOv2/SigLIP/ResNet-50; datasets: mini-imagenet/cifar100/oxford-pets) live in `src/generate_embeddings.py` — import and inspect them below rather than hardcoding names here.

Run this notebook on a Colab GPU runtime (Runtime -> Change runtime type -> GPU).

In [ ]:
import os

REPO_URL = "https://github.com/Nizaxga/eigen-feature.git"
REPO_DIR = "/content/eigen-feature"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## Optional: mount Drive

Persistence — survive a Colab session reset without re-running every (model, dataset, split) combo.

Skip this cell if you don't want Drive access — `outputs/` then stays a plain local directory, wiped on session reset.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Same save_dir convention as distrillation-model-expr's colab notebooks.
save_dir = "/content/drive/MyDrive/eigen-feature"
print(f"[LOG] save_dir={save_dir}")

In [ ]:
import os

# Symlink outputs/ onto Drive so generated embeddings survive a session reset --
# mirrors the outputs/ symlink convention in distrillation-model-expr's colab notebooks.
# Skip if you didn't run the drive.mount() cell above.
DRIVE_OUTPUTS = os.path.join(save_dir, "output-embedding-generation")

if os.path.isdir("/content/drive/MyDrive") and not os.path.exists("outputs"):
    os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
    os.symlink(DRIVE_OUTPUTS, "outputs")
    print(f"[LOG] outputs/ symlinked to {DRIVE_OUTPUTS}")

In [ ]:
import sys

sys.path.append("src")

import torch

from generate_embeddings import DATASET_REGISTRY, MODEL_REGISTRY, run

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[LOG] device={device}")
print("models:", list(MODEL_REGISTRY))
print("datasets:", list(DATASET_REGISTRY))

## Smoke test

Small `max_samples` first — confirms the (model, dataset, split) combo downloads and runs before committing to a full pass.

In [ ]:
run(
    model_name="siglip-base-patch16",
    dataset_name="cifar100",
    split="train",
    batch_size=64,
    max_samples=200,
    output_root="outputs/embeddings",
    device=device,
)

## Full sweep

Edit `COMBOS` to whichever (model, dataset, split) triples you actually want, then run. Each `run()` call is independent — safe to re-run this cell after a session reset, it just re-does whichever combos aren't already saved (no dedup check here; delete the corresponding `.npz` under `outputs/embeddings/` to force a redo, or edit `COMBOS`).

In [ ]:
COMBOS = [
    ("siglip-base-patch16", "cifar100", "train"),
    ("siglip-base-patch16", "cifar100", "test"),
    ("resnet50", "oxford-pets", "train"),
    ("resnet50", "oxford-pets", "test"),
]

for model_name, dataset_name, split in COMBOS:
    print(f"=== {model_name} / {dataset_name} / {split} ===")
    run(
        model_name=model_name,
        dataset_name=dataset_name,
        split=split,
        batch_size=64,
        max_samples=None,
        output_root="outputs/embeddings",
        device=device,
    )

## Done

`outputs/embeddings/<dataset>/<model>_<split>.npz` now holds everything `src/eval_embeddings.py` / `src/eval_online.py` / `src/eval_abtt.py` need. If you symlinked `outputs/` onto Drive, this survives a session reset; otherwise download `outputs/embeddings/` before disconnecting.